# KubeMedic GRPO Training on Hugging Face

This notebook keeps the workflow simple:
- load `HF_TOKEN` from `.env` when available
- install the KubeMedic package plus training extras
- smoke-test the deployed Space
- run the single-session GRPO trainer
- inspect the saved artifacts


In [1]:
import os
import subprocess
from pathlib import Path

os.environ.setdefault('WANDB_DISABLED', 'true')
os.environ.setdefault('TRL_EXPERIMENTAL_SILENCE', '1')

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
env_path = next((path for path in env_candidates if path.exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        if line.startswith('HF_TOKEN=') and not os.environ.get('HF_TOKEN'):
            os.environ['HF_TOKEN'] = line.split('=', 1)[1].strip().strip('"').strip("'")
            break

if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = subprocess.check_output(['hf', 'auth', 'token'], text=True).strip()
    except Exception:
        pass

SPACE_ID = 'ashiqabdulkhader/Kubemedic'
ENV_URL = f"https://{SPACE_ID.replace('/', '-')}.hf.space"
PACKAGE_INSTALL = f"openenv-kubemedic[train] @ git+https://huggingface.co/spaces/{SPACE_ID}.git"
OUTPUT_DIR = Path('outputs/kubemedic-qwen25-3b-grpo')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_EPISODES = 8
EVAL_EPISODES = 2
NUM_GENERATIONS = 2
MAX_COMPLETION_LENGTH = 256
MAX_TURNS = 8

print('HF token loaded:', bool(os.environ.get('HF_TOKEN')))
print('ENV_URL        :', ENV_URL)
print('PACKAGE_INSTALL:', PACKAGE_INSTALL)
print('OUTPUT_DIR     :', OUTPUT_DIR)
print('NUM_GENERATIONS:', NUM_GENERATIONS)


HF token loaded: False
ENV_URL        : https://ashiqabdulkhader-Kubemedic.hf.space
PACKAGE_INSTALL: openenv-kubemedic[train] @ git+https://huggingface.co/spaces/ashiqabdulkhader/Kubemedic.git
OUTPUT_DIR     : outputs/kubemedic-qwen25-3b-grpo
NUM_GENERATIONS: 2


In [2]:
%pip install -q -U "{PACKAGE_INSTALL}"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from Kubemedic import KubemedicAction, KubemedicEnv

env = KubemedicEnv(base_url=ENV_URL).sync()
try:
    result = env.reset(scenario='KUBE-03')
    print(result.observation)
    step_result = env.step(KubemedicAction(tool='kubectl_get', args={'resource': 'pods', 'namespace': 'challenge'}))
    print('\n--- smoke tool call ---\n')
    print(step_result.observation)
finally:
    env.close()


TimeoutError: 

In [ ]:
!train-kubemedic-grpo \
  --env-url "{ENV_URL}" \
  --output-dir "{OUTPUT_DIR}" \
  --train-episodes {TRAIN_EPISODES} \
  --eval-episodes {EVAL_EPISODES} \
  --num-generations {NUM_GENERATIONS} \
  --per-device-train-batch-size {NUM_GENERATIONS} \
  --max-completion-length {MAX_COMPLETION_LENGTH} \
  --max-turns {MAX_TURNS} \
  --vllm-mode colocate


In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(OUTPUT_DIR / 'episode_rewards.csv').tail(10))
print(json.loads((OUTPUT_DIR / 'training_summary.json').read_text()))
display(Image(filename=str(OUTPUT_DIR / 'reward_plot.png')))

trainer_plot = OUTPUT_DIR / 'trainer_metrics.png'
if trainer_plot.exists():
    display(Image(filename=str(trainer_plot)))
